# Common Test I – Multi-Class Classification of Strong Gravitational Lensing Images

**Task:** Build a model for classifying strong lensing images into three classes using PyTorch.

**Classes:**
- `no` – No substructure
- `sphere` – Subhalo substructure  
- `vort` – Vortex substructure

**Evaluation Metrics:** ROC curve and AUC score

---

## Strategy

We use a **ResNet-18** backbone pretrained on ImageNet, adapted for single-channel (grayscale) input. This is a strong choice because:

1. **Transfer learning** – ResNet-18's learned feature hierarchies (edges → textures → shapes) transfer well to lensing images, even though the domain is different. The low-level features (edges, arcs) are universal.
2. **Efficiency** – ResNet-18 is lightweight (~11M params) and trains fast, suitable for this dataset size (30k images).
3. **Skip connections** – Residual connections prevent gradient degradation and enable learning fine-grained differences between substructure types.

**Data augmentation** includes random flips and small rotations, which are physically valid transformations for lensing images (orientation is arbitrary). We avoid aggressive augmentation since the images are already preprocessed.

**Normalization:** Images are already min-max normalized. We standardize to ImageNet statistics to match the pretrained backbone's expected input distribution.

## 1. Setup & Configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix, RocCurveDisplay
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Device Configuration ──────────────────────────────────────────────
# Set DEVICE_OVERRIDE to "cuda", "mps", or "cpu" to force a specific device.
# Set to None for auto-detection.
DEVICE_OVERRIDE = None

if DEVICE_OVERRIDE:
    DEVICE = torch.device(DEVICE_OVERRIDE)
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

# ── Hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE = 64
NUM_EPOCHS = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 3
NUM_WORKERS = 0  # Set >0 if not on MPS/Windows
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Dataset Paths ─────────────────────────────────────────────────────
DATA_ROOT = "dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR = os.path.join(DATA_ROOT, "val")
CLASS_NAMES = ["no", "sphere", "vort"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

print(f"Classes: {CLASS_NAMES}")
print(f"Train dir: {TRAIN_DIR}")
print(f"Val dir:   {VAL_DIR}")

## 2. Dataset & DataLoader

In [ ]:
class LensingDataset(Dataset):
    """Dataset for loading .npy strong lensing images."""

    def __init__(self, root_dir, class_names, transform=None):
        self.samples = []  # list of (path, label)
        self.transform = transform

        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            label = CLASS_TO_IDX[class_name]
            for fname in os.listdir(class_dir):
                if fname.endswith(".npy"):
                    self.samples.append((os.path.join(class_dir, fname), label))

        print(f"  Loaded {len(self.samples)} samples from {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        # Load .npy → shape (1, 150, 150), float64
        img = np.load(path).astype(np.float32)  # (1, 150, 150)

        # Convert to torch tensor
        img = torch.from_numpy(img)  # (1, 150, 150)

        if self.transform:
            img = self.transform(img)

        return img, label


# ── Transforms ─────────────────────────────────────────────────────────
# Images are already min-max normalized [0,1].
# We repeat channels to 3 for pretrained ResNet, then normalize to ImageNet stats.

train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.Lambda(lambda x: x.repeat(3, 1, 1)),  # 1ch → 3ch
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Lambda(lambda x: x.repeat(3, 1, 1)),  # 1ch → 3ch
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ── Create datasets & loaders ─────────────────────────────────────────
print("Loading datasets...")
train_dataset = LensingDataset(TRAIN_DIR, CLASS_NAMES, transform=train_transform)
val_dataset = LensingDataset(VAL_DIR, CLASS_NAMES, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 3. Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, class_name in enumerate(CLASS_NAMES):
    # Load a raw sample (no transform) for visualization
    sample_path = os.path.join(TRAIN_DIR, class_name, "1.npy")
    img = np.load(sample_path)  # (1, 150, 150)
    axes[i].imshow(img[0], cmap="inferno")
    axes[i].set_title(f"Class: {class_name}", fontsize=14)
    axes[i].axis("off")
plt.suptitle("Sample Lensing Images", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Model Definition – ResNet-18

In [ ]:
def build_resnet18(num_classes=3, pretrained=True):
    """Build ResNet-18 adapted for 3-channel replicated grayscale input."""
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)

    # Replace final FC layer for our 3 classes
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model


model = build_resnet18(NUM_CLASSES, pretrained=True).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# ── Training history ──────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
best_model_state = None
patience = 5
patience_counter = 0


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{100.*correct/total:.1f}%")

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="  Val  ", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


# ── Training ──────────────────────────────────────────────────────────
print(f"Training ResNet-18 for {NUM_EPOCHS} epochs on {DEVICE}...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} │ "
          f"Train Loss: {train_loss:.4f}  Acc: {100*train_acc:.1f}% │ "
          f"Val Loss: {val_loss:.4f}  Acc: {100*val_acc:.1f}% │ LR: {lr:.6f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"         ✓ New best val loss!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch} (patience={patience})")
            break

# Restore best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nRestored best model (val_loss={best_val_loss:.4f})")

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_range, history["train_loss"], "o-", label="Train Loss", color="#4C72B0")
ax1.plot(epochs_range, history["val_loss"], "s-", label="Val Loss", color="#DD8452")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, [a*100 for a in history["train_acc"]], "o-", label="Train Acc", color="#4C72B0")
ax2.plot(epochs_range, [a*100 for a in history["val_acc"]], "s-", label="Val Acc", color="#DD8452")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy Curves")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle("ResNet-18 Training History", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Evaluation – ROC Curves & AUC Scores

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    """Get all predictions and true labels from a dataloader."""
    model.eval()
    all_probs = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Predicting", leave=False):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

    return np.concatenate(all_probs), np.concatenate(all_labels)


def plot_roc_curves(y_true, y_probs, class_names, title="ROC Curves"):
    """Plot per-class ROC curves + macro-average."""
    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

    fig, ax = plt.subplots(figsize=(8, 7))
    colors = ["#4C72B0", "#DD8452", "#55A868"]

    # Per-class ROC
    auc_scores = {}
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
        roc_auc = auc(fpr, tpr)
        auc_scores[class_names[i]] = roc_auc
        ax.plot(fpr, tpr, color=colors[i], lw=2,
                label=f"{class_names[i]} (AUC = {roc_auc:.4f})")

    # Macro-average ROC
    all_fpr = np.unique(np.concatenate([roc_curve(y_true_bin[:, i], y_probs[:, i])[0]
                                         for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
        mean_tpr += np.interp(all_fpr, fpr, tpr)
    mean_tpr /= n_classes
    macro_auc = auc(all_fpr, mean_tpr)
    auc_scores["macro"] = macro_auc
    ax.plot(all_fpr, mean_tpr, "k--", lw=2,
            label=f"Macro-average (AUC = {macro_auc:.4f})")

    ax.plot([0, 1], [0, 1], "gray", lw=1, linestyle=":")
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    return auc_scores


# ── Get predictions & plot ────────────────────────────────────────────
val_probs, val_labels = get_predictions(model, val_loader, DEVICE)
val_preds = val_probs.argmax(axis=1)

auc_scores = plot_roc_curves(val_labels, val_probs, CLASS_NAMES,
                              title="ResNet-18 – ROC Curves (Validation Set)")

print("\n" + "="*50)
print("AUC Scores")
print("="*50)
for name, score in auc_scores.items():
    print(f"  {name:>15s}: {score:.4f}")
print("="*50)

## 8. Confusion Matrix & Classification Report

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────
cm = confusion_matrix(val_labels, val_preds)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.figure.colorbar(im, ax=ax)

ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
       ylabel="True Label", xlabel="Predicted Label",
       title="Confusion Matrix")

# Add text annotations
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14)

plt.tight_layout()
plt.show()

# ── Classification Report ─────────────────────────────────────────────
print("\nClassification Report:")
print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES, digits=4))

## 9. Discussion

### Model Choice
ResNet-18 was chosen as the backbone for several reasons:
- **Parameter efficiency**: With ~11M parameters, it's large enough to capture complex features in lensing images but small enough to train quickly on a single GPU/MPS device.
- **Pretrained features**: Even though ImageNet and gravitational lensing are different domains, early-layer features (edges, curves, gradients) transfer well. The arc-like structures in lensing images benefit from these learned primitives.
- **Proven architecture**: Residual connections ensure stable training and good gradient flow.

### Data Handling
- The images are single-channel and already min-max normalized. We replicate to 3 channels and apply ImageNet normalization to match the pretrained backbone's expected distribution.
- Augmentation (flips, rotations) is physically motivated: gravitational lensing has no preferred orientation.
- The dataset is perfectly balanced (10k per class), so no class weighting is needed.

### Expected Results
- Given the balanced classes and clear visual differences between substructure types, we expect AUC > 0.95 for all classes.
- The `no` class may be easiest to distinguish (cleaner ring structure), while `sphere` vs `vort` may be more challenging (both show substructure perturbations).

### Potential Improvements
1. **Larger models**: ResNet-50 or EfficientNet could capture finer details
2. **Test-time augmentation (TTA)**: Average predictions over multiple augmented views
3. **Mixup / CutMix**: Regularization techniques for better generalization
4. **Learning rate warmup**: Gradual warmup before cosine decay